In [1]:
# 03_fairness_metrics.ipynb
# Cross-Domain AI Governance Gap Analysis
# 03. Unified Baseline & Fairness Metrics
#
# Train one unified baseline (XGBoost) on every dataset and compute a common set of
# group-fairness metrics so credit and insurance are directly comparable:
#   Statistical Parity Difference, Disparate Impact, Equal Opportunity Diff, Average Odds Diff.
# Each dataset uses one primary binary sensitive attribute; Porto Seguro uses the
# representative proxy ps_ind_04_cat (remaining proxies analyzed as a supplement).
# Comparability fixes: scale_pos_weight for imbalance + prevalence-based decision threshold.

import os
import json
import warnings
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

DATA_DIR = os.path.join("..", "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
RESULTS_DIR = os.path.join("..", "results")
TAB_DIR = os.path.join(RESULTS_DIR, "tables")
os.makedirs(TAB_DIR, exist_ok=True)

RANDOM_STATE = 42
REPRESENTATIVE_PROXY = "ps_ind_04_cat"

with open(os.path.join(PROCESSED_DIR, "preprocess_meta.json")) as f:
    META = json.load(f)

# (display_name, key, primary sensitive column, privileged value; None -> most frequent)
DATASETS = [
    ("German Credit",        "german",               "sex",                "male"),
    ("Credit Card Approval", "credit_card_approval",  "sex",                "male"),
    ("Medical Cost",         "insurance",             "sex",                "male"),
    ("Porto Seguro",         "porto_seguro",          REPRESENTATIVE_PROXY, None),
]

def load_split(key):
    return (pd.read_parquet(os.path.join(PROCESSED_DIR, f"{key}_train.parquet")),
            pd.read_parquet(os.path.join(PROCESSED_DIR, f"{key}_test.parquet")))

def get_Xy(df, key):
    meta = META[key]
    target_col = meta["target_col"]
    sensitive_cols = meta["sensitive_cols"]
    aux_cols = [c for c in ["target_60dpd", "target_30dpd"] if c in df.columns and c != target_col]
    drop = [target_col] + sensitive_cols + aux_cols
    X = df.drop(columns=[c for c in drop if c in df.columns], errors="ignore")
    return X, df[target_col].astype(int)

# Positive prediction = predicted high-risk (label 1)
def fairness_metrics(y_true, y_pred, priv_mask, unpriv_mask):
    def rates(mask):
        yt, yp = y_true[mask], y_pred[mask]
        ppr = yp.mean() if len(yp) else np.nan
        pos, neg = yt == 1, yt == 0
        tpr = yp[pos].mean() if pos.sum() else np.nan
        fpr = yp[neg].mean() if neg.sum() else np.nan
        return ppr, tpr, fpr
    ppr_p, tpr_p, fpr_p = rates(priv_mask)
    ppr_u, tpr_u, fpr_u = rates(unpriv_mask)
    spd = ppr_u - ppr_p
    di = (ppr_u / ppr_p) if ppr_p not in (0, np.nan) else np.nan
    eod = tpr_u - tpr_p
    aod = 0.5 * ((fpr_u - fpr_p) + (tpr_u - tpr_p))
    return {"StatParityDiff": spd, "DisparateImpact": di, "EqualOppDiff": eod, "AvgOddsDiff": aod}

rows, pred_store = [], {}
for display_name, key, sens_col, privileged in DATASETS:
    train, test = load_split(key)
    X_train, y_train = get_Xy(train, key)
    X_test, y_test = get_Xy(test, key)
    X_test = X_test[X_train.columns]
    n_pos, n_neg = int(y_train.sum()), int((y_train == 0).sum())
    spw = (n_neg / n_pos) if n_pos > 0 else 1.0
    model = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9, scale_pos_weight=spw,
        eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1)
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    prevalence = float(y_train.mean())
    thr = np.quantile(proba, 1.0 - prevalence)
    y_pred = (proba >= thr).astype(int)
    s = test[sens_col]
    if privileged is None:
        privileged = s.mode(dropna=True).iloc[0]
    priv_mask = (s == privileged).values
    unpriv_mask = (s != privileged).values & s.notna().values
    fm = fairness_metrics(y_test.values, y_pred, priv_mask, unpriv_mask)
    fm.update({"domain": META[key].get("domain", ""), "dataset": display_name,
               "sensitive": sens_col, "privileged": str(privileged), "threshold": round(thr, 4),
               "pred_pos_rate": round(float(y_pred.mean()), 4),
               "AUC": roc_auc_score(y_test.values, proba),
               "PR_AUC": average_precision_score(y_test.values, proba)})
    rows.append(fm)
    keep = list(dict.fromkeys(list(META[key]["sensitive_cols"]) + [sens_col]))
    pred_df = test[keep].copy()
    pred_df["y_true"] = y_test.values
    pred_df["proba"] = proba
    pred_df["y_pred"] = y_pred
    pred_store[key] = pred_df
    print(f"[{display_name:>22}] AUC={fm['AUC']:.3f}  DI={fm['DisparateImpact']:.3f}  "
          f"SPD={fm['StatParityDiff']:+.3f}")

fair_df = pd.DataFrame(rows)[[
    "domain", "dataset", "sensitive", "privileged", "threshold", "pred_pos_rate",
    "AUC", "PR_AUC", "StatParityDiff", "DisparateImpact", "EqualOppDiff", "AvgOddsDiff"]].round(4)
print(fair_df.to_string(index=False))
fair_df.to_csv(os.path.join(TAB_DIR, "03_fairness_primary.csv"), index=False)

# Supplement: Porto Seguro remaining proxies under the same metrics
porto_key = "porto_seguro"
porto_test = load_split(porto_key)[1]
porto_pred = pred_store[porto_key]
supp_rows = []
for pc in META[porto_key]["sensitive_cols"]:
    s = porto_test[pc]
    if s.nunique(dropna=True) < 2:
        continue
    privileged = s.mode(dropna=True).iloc[0]
    priv_mask = (s == privileged).values
    unpriv_mask = (s != privileged).values & s.notna().values
    fm = fairness_metrics(porto_pred["y_true"].values, porto_pred["y_pred"].values,
                          priv_mask, unpriv_mask)
    fm.update({"proxy": pc, "privileged": str(privileged),
               "is_representative": pc == REPRESENTATIVE_PROXY})
    supp_rows.append(fm)
supp_df = pd.DataFrame(supp_rows)[[
    "proxy", "privileged", "is_representative",
    "StatParityDiff", "DisparateImpact", "EqualOppDiff", "AvgOddsDiff"]].round(4)
print(supp_df.to_string(index=False))
supp_df.to_csv(os.path.join(TAB_DIR, "03_porto_proxy_supplement.csv"), index=False)
print("03_fairness_metrics complete")


[         German Credit] AUC=0.812  DI=1.170  SPD=+0.048


[  Credit Card Approval] AUC=0.683  DI=0.773  SPD=-0.032
[          Medical Cost] AUC=0.574  DI=0.971  SPD=-0.007


[          Porto Seguro] AUC=0.568  DI=0.971  SPD=-0.001
   domain              dataset     sensitive privileged  threshold  pred_pos_rate    AUC  PR_AUC  StatParityDiff  DisparateImpact  EqualOppDiff  AvgOddsDiff
   credit        German Credit           sex       male     0.4670         0.3000 0.8120  0.6657          0.0484           1.1701        0.0671       0.0331
   credit Credit Card Approval           sex       male     0.5729         0.1177 0.6835  0.2671         -0.0315           0.7732       -0.0557      -0.0404
insurance         Medical Cost           sex       male     0.5620         0.2507 0.5742  0.3161         -0.0074           0.9709       -0.0276      -0.0132
insurance         Porto Seguro ps_ind_04_cat        0.0     0.5939         0.0365 0.5678  0.0478         -0.0011           0.9715       -0.0468      -0.0230
        proxy privileged  is_representative  StatParityDiff  DisparateImpact  EqualOppDiff  AvgOddsDiff
ps_ind_02_cat        1.0              False          0